# Simplest version — `re` does all the work

Two imports only: `re` for every bit of parsing, filtering, cleaning and file writing, and
`urllib.request` because something has to download the page. **No `csv`, no `os`, no `json`,
no `pandas`, no BeautifulSoup.**

It collects **name, e-mail and room number** for the Faculty of Science & Technology and saves
them as CSV. The feed URL below is the one the faculty page's own JavaScript loads (the site
builds its cards with JS, so the page HTML contains no data at all).

In [1]:
import re
import urllib.request

URL = ("https://www.aiub.edu/Files/Uploads/public-employee-profiles/"
       "employeeProfiles.json?v=1.0.11")
request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
text = urllib.request.urlopen(request, timeout=60).read().decode("utf-8", "replace")

# Regex 1: cut the text right before every employee object
RECORDS = re.compile(r'(?=\{"IsShowProfileDetails)')
records = [r for r in RECORDS.split(text) if r.startswith('{"IsShow')]

print("characters downloaded :", len(text))
print("employee records      :", len(records))

characters downloaded : 294863
employee records      : 446


In [2]:
# Regex 2-5: one pattern per thing we need
NAME  = re.compile(r'"Name":"([^"]*)"')
EMAIL = re.compile(r'"Email":"([^"]*)"')
ROOM  = re.compile(r'"RoomNo":"([^"]*)"')
FST   = re.compile(r'"Faculty":"FACULTY OF SCIENCE & TECHNOLOGY"')

def grab(pattern, record):
    match = pattern.search(record)
    return match.group(1) if match else ""

rows = [(grab(NAME, r), grab(EMAIL, r), grab(ROOM, r))
        for r in records if FST.search(r)]

print("science & technology faculty:", len(rows))
print()
for name, email, room in rows[:10]:
    print("{:<30} {:<26} {}".format(name, email, room or "-"))

science & technology faculty: 196

MD. ANWARUL KABIR              kabir@aiub.edu             DNO -216
MASHIOUR RAHMAN                mashiour@aiub.edu          DNGA05
SALMA BEGOM                    salma@aiub.edu             817
DR. MOHAMMED JASHIM UDDIN      drjashim@aiub.edu          1122
PROF. DR. KH. ABDUL MALEQUE    maleque@aiub.edu           DN0915
DR. SARFUDDIN AHMED TAREK      satarek@aiub.edu           DN0722
DR. AFROZA NAHAR               afroza@aiub.edu            DN0113
DR. S. A. M. MANZUR H. KHAN    manzur@aiub.edu            C
DR. FARZANA SABETH             fsabeth@aiub.edu           DN0721
DR. S. MOSADDEQ AHMED          smahmed@aiub.edu           DN0816


In [3]:
# Regex 6: a field only needs quoting when it holds a comma, a quote or a newline
NEEDS_QUOTES = re.compile(r'[",\n]')

def to_csv_field(value):
    return '"{}"'.format(value.replace('"', '""')) if NEEDS_QUOTES.search(value) else value

lines = ["name,email,room"]
lines += [",".join(to_csv_field(v) for v in row) for row in rows]

# builtin open() instead of the csv module
with open("output/simple_faculty.csv", "w", encoding="utf-8") as handle:
    handle.write("\n".join(lines) + "\n")

with open("output/simple_faculty.csv", encoding="utf-8") as handle:
    saved = handle.read()

# Regex 7: count what was written. Every line ends with \n, so counting \n is the line count.
print("saved lines        :", len(re.findall(r"\n", saved)))
print("faculty rows       :", len(re.findall(r"\n", saved)) - 1, "(header row not counted)")
print("quoted fields      :", len(re.findall(r'"[^"]*,[^"]*"', saved)))
bad = [email for _, email, _ in rows if not re.match(
        r"^[\w.+-]+@[\w-]+(\.[\w-]+)+$", email)]
print("e-mails off pattern:", len(bad), bad)
print("written to         : output/simple_faculty.csv")

saved lines        : 197
faculty rows       : 196 (header row not counted)
quoted fields      : 0
e-mails off pattern: 0 []
written to         : output/simple_faculty.csv


That is the entire program: 7 patterns, ~40 lines including comments, one output file, and the
only import besides `re` is `urllib.request`.

What it deliberately leaves out, all of which `AIUB_Faculty_Regex_Scraper.ipynb` covers:

* values stay exactly as AIUB stores them, so names are ALL CAPS (`MD. ANWARUL KABIR`) and room
  codes stay messy (`DNO -216`, `1122 (OR)`);
* the feed URL is typed in by hand instead of being regex-discovered from the page and its script;
* no building, position or department columns, and no per-field sanity report.

The saved CSV is still a valid file: a field gets quoted only when Regex 6 finds a comma, a
quote or a newline inside it, which is the same rule the `csv` module follows.